### News

Placeholder

### Installation

In [ ]:
# Placeholder

### Unsloth

**EmbeddingGemma 2** maps **text, images, audio, video and mixtures of them** into one 768-dimensional
space, so "find the photo / sound / clip that matches this sentence" is a single dot product.

In this notebook we load it with Unsloth and build small search engines over real benchmark data:

| Section | Data | What you get |
|---|---|---|
| Text | a handful of passages | query / document prompts, titles |
| Images | Flickr30k 1K test (Karpathy split) | text->image and image->text search + Recall@K |
| Matryoshka | same | 768 / 512 / 256 / 128-dim embeddings, quality vs size |
| Audio | Clotho (MTEB test split) | text->sound search with players + Recall@K |
| Video | MSR-VTT 1K-A test | text->video search |
| Interleaved | text + image + audio in one query | one vector for a mixed query |
| Partial loading | none | load only the towers you need, VRAM table |

On a Tesla T4 (no bfloat16) Unsloth runs the model in float32 automatically: EmbeddingGemma 2 does not
support float16 activations.

In [ ]:
# EmbeddingGemma 2 may need a Hugging Face token (Colab: add HF_TOKEN under Secrets).
import os
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

In [ ]:
from unsloth import FastSentenceTransformer
import torch

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/embeddinggemma-2",
    for_inference = True, # inference only: no LoRA, no training patches
)
print(model)
print("dtype:", next(model.parameters()).dtype)

In [ ]:
# @title Small display helpers (thumbnails, audio players, video)
import io, base64, html, torch
from IPython.display import display, HTML, Audio, Video
from PIL import Image

def show_images(images, captions = None, size = 160):
    cells = []
    for i, img in enumerate(images):
        buf = io.BytesIO(); img.copy().convert("RGB").resize((size, size)).save(buf, format = "JPEG")
        cap = html.escape(captions[i]) if captions else ""
        cells.append(f'<div style="display:inline-block;margin:4px;width:{size}px;font-size:11px;vertical-align:top">'
                     f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"><br>{cap}</div>')
    display(HTML("".join(cells)))

def recall_at_k(similarity, positives, ks = (1, 5, 10)):
    """similarity: [queries, items]; positives[q] = set of correct item indices."""
    ranking = similarity.argsort(dim = 1, descending = True)
    out = {}
    for k in ks:
        top = ranking[:, :k].tolist()
        out[f"R@{k}"] = round(100 * sum(len(set(t) & positives[q]) > 0 for q, t in enumerate(top)) / len(top), 2)
    return out

### 1. Text search with task prompts
EmbeddingGemma 2 was trained with **task prompts**. Queries and documents use different ones:
* query: `task: search result | query: ...` (`prompt_name = "query"`)
* document: `title: none | text: ...` (`prompt_name = "document"`), or `title: {title} | text: {body}` if you have a title.

In [ ]:
documents = [
    "The aurora borealis is caused by charged particles from the solar wind colliding with oxygen and nitrogen in the upper atmosphere.",
    "Photosynthesis lets plants turn carbon dioxide and water into sugar using sunlight.",
    "The Great Barrier Reef is the world's largest coral reef system, off the coast of Queensland, Australia.",
    "Mount Everest's summit is 8,849 metres above sea level.",
    "A transformer is a neural network architecture built on self-attention.",
    "Unsloth makes fine-tuning large language models up to 2x faster with less memory.",
]
query = "Why is the sky green and purple near the north pole at night?"

doc_embeddings = model.encode(documents, prompt_name = "document", convert_to_tensor = True)
query_embedding = model.encode(query, prompt_name = "query", convert_to_tensor = True)
scores = model.similarity(query_embedding, doc_embeddings)[0]
for i in scores.argsort(descending = True).tolist():
    print(f"{scores[i]:.3f} | {documents[i]}")

**Task prompt cheat sheet.** Pass any of these names as `prompt_name`:

| Use case | Query side | Document side |
|---|---|---|
| Search / RAG | `SearchQuery` (= `query`) | `Document` (= `document`) |
| Question answering | `QuestionAnswering` | `Document` |
| Fact checking | `FactChecking` | `Document` |
| Code search | `CodeRetrieval` | `Document` |
| Sentence similarity (both sides) | `SentenceSimilarity` / `STS` | same |
| Clustering | `Clustering` | same |
| Classification | `Classification` | same |

Images, audio and video do not need a prompt.

In [ ]:
for name, prompt in model.prompts.items():
    print(f"{name:26s} -> {prompt!r}")

### 2. Image search on Flickr30k (1K test images, 5K captions)
This is the standard Karpathy test split used to report image-text retrieval.

In [ ]:
import zipfile, csv, json, os
from huggingface_hub import hf_hub_download

repo = "nlphuji/flickr_1k_test_image_text_retrieval"
zip_path = hf_hub_download(repo, "images_flickr_1k_test.zip", repo_type = "dataset")
csv_path = hf_hub_download(repo, "test_1k_flickr.csv", repo_type = "dataset")

rows = list(csv.DictReader(open(csv_path)))
archive = zipfile.ZipFile(zip_path)
name_to_member = {os.path.basename(n): n for n in archive.namelist() if n.endswith(".jpg")}
flickr_images = [Image.open(archive.open(name_to_member[r["filename"]])).convert("RGB") for r in rows]
flickr_captions, caption_to_image = [], []
for i, r in enumerate(rows):
    for c in json.loads(r["raw"]):
        flickr_captions.append(c); caption_to_image.append(i)
print(len(flickr_images), "images,", len(flickr_captions), "captions")
show_images(flickr_images[:6], [json.loads(r["raw"])[0] for r in rows[:6]])

In [ ]:
import time
start = time.time()
image_embeddings = model.encode(flickr_images, batch_size = 16, convert_to_tensor = True, show_progress_bar = True)
caption_embeddings = model.encode(flickr_captions, batch_size = 64, convert_to_tensor = True)
print(f"Embedded {len(flickr_images)} images + {len(flickr_captions)} captions in {time.time() - start:.0f}s")

def search_images(text, k = 5):
    q = model.encode(text, convert_to_tensor = True)
    scores = model.similarity(q, image_embeddings)[0]
    top = scores.topk(k)
    show_images([flickr_images[i] for i in top.indices.tolist()], [f"{s:.3f}" for s in top.values.tolist()])

search_images("a dog jumping to catch a frisbee")
search_images("people sitting at a cafe on a busy street")

In [ ]:
# Image -> image: find visually similar photos (the query image comes first)
scores = model.similarity(image_embeddings[7:8], image_embeddings)[0]
show_images([flickr_images[i] for i in scores.topk(5).indices.tolist()])

**Benchmark.** Recall@K = how often the correct item is in the top K.

In [ ]:
text_to_image = model.similarity(caption_embeddings, image_embeddings).float().cpu()
t2i = recall_at_k(text_to_image, [{caption_to_image[q]} for q in range(len(flickr_captions))])
image_to_texts = [set(j for j, img in enumerate(caption_to_image) if img == i) for i in range(len(flickr_images))]
i2t = recall_at_k(text_to_image.T, image_to_texts)
print("Flickr30k 1K  text->image:", t2i)
print("Flickr30k 1K  image->text:", i2t)

### 3. Matryoshka: smaller vectors, almost the same quality
EmbeddingGemma 2 is trained so the **first 512, 256 or 128 numbers** of each vector are themselves a good
embedding. Use `truncate_dim` to store 1.5x-6x less.

In [ ]:
import torch.nn.functional as F
print(f"{'dims':>5s}  {'text->image R@1':>16s}  {'R@5':>6s}")
for dims in [768, 512, 256, 128]:
    q = F.normalize(caption_embeddings[:, :dims].float(), dim = -1)
    d = F.normalize(image_embeddings[:, :dims].float(), dim = -1)
    r = recall_at_k((q @ d.T).cpu(), [{caption_to_image[i]} for i in range(len(flickr_captions))], ks = (1, 5))
    print(f"{dims:5d}  {r['R@1']:16.2f}  {r['R@5']:6.2f}")

# Same thing directly from encode:
small = model.encode(["a dog on the beach"], truncate_dim = 256, normalize_embeddings = True)
print("truncated shape:", small.shape)

### 4. Sound search on Clotho
Clotho is an audio-captioning benchmark of 15-30 second environmental sounds, each described by 5 people.
We use one shard of the MTEB test split (209 clips).

In [ ]:
import soundfile as sf, numpy as np, pyarrow.parquet as pq
clotho_path = hf_hub_download("mteb/Clotho", "data/test-00000-of-00005.parquet", repo_type = "dataset")
clotho = pq.read_table(clotho_path).to_pylist()

os.makedirs("clotho", exist_ok = True)
clips = [] # file paths: sentence-transformers decodes (and resamples) audio files itself
for r in clotho:
    path = f"clotho/{r['index'].rsplit('_', 1)[-1]}.wav"
    open(path, "wb").write(r["audio"]["bytes"]); clips.append(path)
sound_captions, caption_to_clip = [], []
for i, r in enumerate(clotho):
    for c in [c.strip() + "." for c in r["text"].split(". ") if c.strip()]:
        sound_captions.append(c); caption_to_clip.append(i)
print(len(clips), "clips,", len(sound_captions), "captions")

clip_embeddings = model.encode([{"audio": c} for c in clips], batch_size = 8, convert_to_tensor = True, show_progress_bar = True)
sound_caption_embeddings = model.encode(sound_captions, batch_size = 64, convert_to_tensor = True)

def search_sounds(text, k = 2):
    q = model.encode(text, convert_to_tensor = True)
    scores = model.similarity(q, clip_embeddings)[0]
    for s, i in zip(*scores.topk(k)):
        print(f"{s:.3f} | {clotho[i]['text'].split('. ')[0]}")
        display(Audio(filename = clips[i]))

search_sounds("heavy rain falling on a roof")
r = recall_at_k(model.similarity(sound_caption_embeddings, clip_embeddings).float().cpu(),
                [{caption_to_clip[q]} for q in range(len(sound_captions))])
print("Clotho (209 clips) text->audio:", r)

### 5. Video search on MSR-VTT
MSR-VTT 1K-A is the standard text-to-video test set. We only fetch the first 40 test videos here
(reading them straight out of the remote zip), to keep the download small.

In [ ]:
from huggingface_hub import HfFileSystem
test_1k = json.load(open(hf_hub_download("friedrichor/MSR-VTT", "msrvtt_test_1k.json", repo_type = "dataset")))[:40]
os.makedirs("msrvtt", exist_ok = True)
with HfFileSystem().open("datasets/friedrichor/MSR-VTT/MSRVTT_Videos.zip", "rb") as f:
    remote = zipfile.ZipFile(f)
    members = {os.path.basename(n): n for n in remote.namelist() if n.endswith(".mp4")}
    for item in test_1k:
        target = f"msrvtt/{item['video']}"
        if not os.path.exists(target):
            open(target, "wb").write(remote.read(members[item["video"]]))
videos = [f"msrvtt/{item['video']}" for item in test_1k]
video_captions = [item["caption"] for item in test_1k]

video_embeddings = model.encode([{"video": v} for v in videos], batch_size = 2, convert_to_tensor = True, show_progress_bar = True)
video_caption_embeddings = model.encode(video_captions, convert_to_tensor = True)
r = recall_at_k(model.similarity(video_caption_embeddings, video_embeddings).float().cpu(), [{i} for i in range(len(videos))], ks = (1, 5))
print("MSR-VTT (first 40 of 1K-A) text->video:", r)

query = "fish swimming around in an aquarium"
best = model.similarity(model.encode(query, convert_to_tensor = True), video_embeddings)[0].argmax().item()
print("Query:", query, "| best match:", videos[best], "| its caption:", video_captions[best])
display(Video(videos[best], embed = True, width = 320))

### 6. Interleaved queries: text + images + audio in one vector
Put `<|image|>`, `<|audio|>`, `<|video|>` placeholders in the text to choose where each item goes.

In [ ]:
mixed_query = {
    "text": "A photo like <|image|> but at the beach, with a sound like <|audio|>",
    "image": flickr_images[7],
    "audio": clips[0],
}
mixed = model.encode(mixed_query, convert_to_tensor = True)
print("one vector:", tuple(mixed.shape), "norm", round(mixed.norm().item(), 4))
scores = model.similarity(mixed, image_embeddings)[0]
show_images([flickr_images[i] for i in scores.topk(5).indices.tolist()])

### 7. Only load what you need
Drop the vision and/or audio towers at load time with `config_kwargs`. Text-only is **271M parameters**
instead of 744M.

In [ ]:
import gc
del model; gc.collect(); torch.cuda.empty_cache()
variants = {
    "text only":      {"vision_config": None, "audio_config": None},
    "text + vision":  {"audio_config": None},
    "text + audio":   {"vision_config": None},
    "full (omni)":    {},
}
print(f"{'variant':16s} {'params':>10s} {'VRAM (GB)':>10s}")
for name, config_kwargs in variants.items():
    torch.cuda.empty_cache(); base = torch.cuda.memory_allocated()
    m = FastSentenceTransformer.from_pretrained("unsloth/embeddinggemma-2", for_inference = True, config_kwargs = config_kwargs)
    params = sum(p.numel() for p in m.parameters())
    print(f"{name:16s} {params / 1e6:9.1f}M {(torch.cuda.memory_allocated() - base) / 2**30:10.2f}")
    del m; gc.collect(); torch.cuda.empty_cache()

### Next steps
* Fine-tune EmbeddingGemma 2 on your own text pairs: `EmbeddingGemma2_(300M).ipynb`
* Fine-tune image <-> text retrieval: `EmbeddingGemma2_(300M)-Image_Text.ipynb`
* Fine-tune audio <-> text retrieval: `EmbeddingGemma2_(300M)-Audio.ipynb`